## 7.1 The Dataset

In this exercise, you will train a model to recognize fresh and rotten fruits. The dataset comes from [Kaggle](https://www.kaggle.com/sriramr/fruits-fresh-and-rotten-for-classification), a great place to go if you're interested in starting a project after this class. The dataset structure is in the `data/fruits` folder. There are 6 categories of fruits: fresh apples, fresh oranges, fresh bananas, rotten apples, rotten oranges, and rotten bananas. This will mean that your model will require an output layer of 6 neurons to do the categorization successfully. You'll also need to compile the model with `categorical_crossentropy`, as we have more than two categories.

In [3]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms.v2 as transforms
import torchvision.io as tv_io
import os

from torch.utils.data import random_split

import glob
from PIL import Image

import training_utils_py as utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

ModuleNotFoundError: No module named 'training_utils_py'

## 7.2 Load ImageNet Base Model

We encourage you to start with a model pretrained on ImageNet. Load the model with the correct weights. Because these pictures are in color, there will be three channels for red, green, and blue. We've filled in the input shape for you. 

In [ ]:
from torchvision.models import vgg16
from torchvision.models import VGG16_Weights

#weights = VGG16_Weights.FIXME
weights = VGG16_Weights.DEFAULT
vgg_model = vgg16(weights=weights)

## 7.3 Freeze Base Model

Next, we suggest freezing the base model, as done in [notebook 05b](05b_presidential_doggy_door.ipynb). This is done so that all the learning from the ImageNet dataset does not get destroyed in the initial training.

In [ ]:
# Freeze base model
#vgg_model.requires_grad_(FIXME)
vgg_model.requires_grad_(False)
next(iter(vgg_model.parameters())).requires_grad

## 7.4 Add Layers to Model

Now it's time to add layers to the pretrained model. Pay close attention to the last dense layer and make sure it has the correct number of neurons to classify the different types of fruit.

The later layers of a model become more specific to the data the model trained on. Since we want the more general learnings from VGG, we can select parts of it, like so:

In [ ]:
vgg_model.classifier[0:3]

Once we've taken what we've wanted from VGG16, we can then add our own modifications. No matter what additional modules we add, we still need to end with one value for each output.

In [ ]:
#N_CLASSES = FIXME
N_CLASSES = 6

my_model = nn.Sequential(
    vgg_model.features,
    vgg_model.avgpool,
    nn.Flatten(),
    vgg_model.classifier[0:3],
    nn.Linear(4096, 500),
    nn.ReLU(),
    nn.Linear(500, N_CLASSES)
)
my_model

## 7.5 Compile Model

Now it's time to compile the model with loss and metrics options. We have 6 classes, so which loss function should we use?

In [ ]:
#loss_function = nn.FIXME()
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(my_model.parameters())
my_model = torch.compile(my_model.to(device))

## 7.6 Data Transforms

To preprocess our input images, we will use the transforms included with the VGG16 weights.

In [ ]:
pre_trans = weights.transforms()

Try to randomly augment the data to improve the dataset.  There is also documentation for the [TorchVision Transforms class](https://pytorch.org/vision/stable/transforms.html).

**Hint**: Remember not to make the data augmentation too extreme.

In [ ]:
IMG_WIDTH, IMG_HEIGHT = (224, 224)

#random_trans = transforms.Compose([
#    FIXME
#])

train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    #transforms.RandomHorizontalFlip(),
    #transforms.RandomRotation(10),
    transforms.ConvertImageDtype(torch.float32)
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ConvertImageDtype(torch.float32)
])

## 7.7 Load Dataset

Now it's time to load the train and validation datasets. 

In [ ]:
DATA_LABELS = [
    "freshapples",
    "freshbanana",
    "freshoranges",
    "rottenapples",
    "rottenbanana",
    "rottenoranges"
]

class MyDataset(Dataset):
    
    def __init__(self, data_dir, transform=None):
        self.paths = []
        self.labels = []
        self.transform = transform
        
        for l_idx, label in enumerate(DATA_LABELS):
            data_paths = glob.glob(data_dir + label + '/*.png')
            
            for path in data_paths:
                self.paths.append(path)
                self.labels.append(l_idx)

    def __getitem__(self, idx):

        img = tv_io.read_image(self.paths[idx], tv_io.ImageReadMode.RGB)

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(self.labels[idx])

        return img, label

    def __len__(self):
        return len(self.paths)

Select the batch size `n` and set `shuffle` either to `True` or `False` depending on if we are `train`ing or `valid`ating.

In [ ]:
train_path = "/kaggle/input/datasets/sriramr/fruits-fresh-and-rotten-for-classification/dataset/train/"
train_dataset_full = MyDataset(train_path, transform=train_tf)
val_dataset_full   = MyDataset(train_path, transform=val_tf)

N = len(train_dataset_full)

train_size = int(0.8 * N)
val_size = N - train_size

g = torch.Generator().manual_seed(42)
perm = torch.randperm(N, generator=g)

train_idx = perm[:train_size]
val_idx = perm[train_size:]

train_dataset = Subset(train_dataset_full, train_idx)
val_dataset   = Subset(val_dataset_full, val_idx)

test_path = "/kaggle/input/datasets/sriramr/fruits-fresh-and-rotten-for-classification/dataset/test/"
test_dataset = MyDataset(test_path, transform=val_tf)


In [ ]:
torch.backends.cudnn.benchmark = True  # acelera convs con tamaños fijos

nw = min(4, os.cpu_count())  # Kaggle suele ir bien con 2–4

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=nw,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

valid_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=nw,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=nw,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

In [ ]:
train_N = len(train_loader.dataset)
valid_N = len(val_loader.dataset)
test_N = len(test_loader.dataset)

## 8 Train the Model

Time to train the model! We've moved the `train` and `validate` functions to our [utils.py](./utils.py) file. Before running the below, make sure all your variables are correctly defined.

It may help to rerun this cell or change the number of `epochs`.

In [ ]:
epochs = 15

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    utils.train(my_model, train_loader, train_N, None, optimizer, loss_function, device)
    utils.validate(my_model, valid_loader, loss_function, device)

## 9 Unfreeze Model for Fine Tuning

If you have reached 92% validation accuracy already, this next step is optional. If not, we suggest fine tuning the model with a very low learning rate.

In [ ]:
# Unfreeze the base model
vgg_model.requires_grad_(True)
optimizer = Adam(my_model.parameters(), lr=.0001)

In [ ]:
epochs = 5

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    utils.train(my_model, train_loader, train_N, random_trans, optimizer, loss_function)
    utils.validate(my_model, valid_loader, valid_N, loss_function)

## 10 Evaluate the Model

Hopefully, you now have a model that has a validation accuracy of 92% or higher. If not, you may want to go back and either run more epochs of training, or adjust your data augmentation. 

Once you are satisfied with the validation accuracy, evaluate the model by executing the following cell. The evaluate function will return a tuple, where the first value is your loss, and the second value is your accuracy. To pass, the model will need have an accuracy value of `92% or higher`. 

In [ ]:
utils.validate(my_model, test_loader, loss_function, device)